# 任意组合扫描：只读分析

读取新的组合 SQLite，以及旧 Temperature×Lock-in / Three-SMU 记录；不连接仪器。

采集循环顺序不限制 X/Y/分组。默认只选择 accepted/clean；原始记录不修改。新组合执行器目前仅产生 **simulation** 数据。重复点、segment、direction、各仪器时间戳保留；分组不等于重新执行了另一种扫描顺序。


In [ ]:
from pathlib import Path
import sys
import sqlite3
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "attodry_control").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "attodry_control").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from attodry_control.combination_analysis import (
    load_combination_rows, load_legacy_three_smu, load_legacy_temperature_lockin,
    select_series, plot_series, export_selection,
)
from IPython.display import display
import ipywidgets as widgets

DATA_DIRECTORY = PROJECT_ROOT / "run_data"  # Set once; local or SSH-kernel path.


## 选择记录

Refresh records → 多选记录 → Load selected records。SQLite 按运行标记过滤；旧温度—激励沿用其已完成 condition 规则。Audit 可以查看新 SQLite / Three-SMU 非接受 formal 样本，但旧温度适配仍只读取已完成 clean 样本；原始 rejected/partial 证据留在各自审计文件。


In [ ]:
files = widgets.SelectMultiple(options=[], description="Records:", rows=8, layout=widgets.Layout(width="95%"))
refresh_button = widgets.Button(description="Refresh records")
load_button = widgets.Button(description="Load selected records")
audit_box = widgets.Checkbox(value=False, description="Audit (include non-accepted)")
messages = widgets.Output()
excluded = widgets.SelectMultiple(options=[], description="Exclude:", rows=6, layout=widgets.Layout(width="95%"))
loaded_rows = []
loaded_audit = False

def sample_id(row):
    return (row.get("source_path"), row.get("run_id"), row.get("condition_id"),
            row.get("attempt_index"), row.get("sample_index"),
            row.get("role"), row.get("harmonic"))

def refresh_records(_=None):
    choices = []
    if DATA_DIRECTORY.exists():
        choices.extend((str(path.relative_to(DATA_DIRECTORY)), ("combined", str(path)))
                       for path in DATA_DIRECTORY.rglob("*.sqlite"))
        choices.extend((str(path.parent.relative_to(DATA_DIRECTORY)), ("smu", str(path.parent)))
                       for path in DATA_DIRECTORY.rglob("metadata.json")
                       if (path.parent / "data.csv").is_file())
        from attodry_control.temperature_excitation_analysis import discover_temperature_excitation_records
        choices.extend((str(path.relative_to(DATA_DIRECTORY)), ("temperature_lockin", str(path)))
                       for path in discover_temperature_excitation_records(DATA_DIRECTORY))
    files.options = sorted(choices)
    with messages:
        messages.clear_output()
        print(f"{len(choices)} record sources; nothing has been connected.")

def load_selected(_=None):
    global loaded_rows, loaded_audit
    loaded_rows = []
    loaded_audit = audit_box.value
    with messages:
        messages.clear_output()
        for kind, path in files.value:
            try:
                if kind == "combined":
                    loaded_rows.extend(load_combination_rows(path, audit=loaded_audit))
                elif kind == "smu":
                    loaded_rows.extend(load_legacy_three_smu(path, audit=loaded_audit))
                else:
                    loaded_rows.extend(load_legacy_temperature_lockin(path))
                    if loaded_audit:
                        print("Legacy temperature adapter remains completed/clean-only:", path)
            except (ValueError, OSError, sqlite3.DatabaseError) as error:
                print(path, error)
        # Exact IDs include source/attempt/sample/role/harmonic; no coordinate dedup.
        identities = list(dict.fromkeys(sample_id(row) for row in loaded_rows))
        excluded.options = [(str(key), key) for key in identities]
        excluded.value = ()
        print(f"Loaded {len(loaded_rows)} rows. Columns:")
        print("\n".join(sorted({key for row in loaded_rows for key in row})))

refresh_button.on_click(refresh_records)
load_button.on_click(load_selected)
display(widgets.VBox([widgets.HBox([refresh_button, load_button, audit_box]), files, excluded, messages]))
refresh_records()


## 分析选择

横轴下拉框可选择估算的 Lock-in 电流或实测 SINE OUT 电压；默认 Custom X 使用下方 X 变量。切换后重新运行本 cell，再导出。

示例：SMU 为外层，Lock-in 激励为内层；下方仍按激励分组查看 SMU I–V。X/Y 使用实测值；激励分组使用 requested 值，避免仪器微小读回波动把同一条件拆开。`FILTERS` 是精确列值匹配，不进行隐式温度/磁场分箱。

函数额外保护 run/repeat/sample/segment/direction 及未绘制坐标；返回的分组标签列出这些信息。默认按采集顺序保存原始点，不做平均、插值或透视填零。图为散点，以免跨过缺失数据连线。


In [ ]:
X = "measured.smu_bias_voltage_v"
if 'excitation_x_widget' not in globals():
    excitation_x_widget = widgets.Dropdown(
        options=(('Custom X below', None),
                 ('Calculated lock-in current (A RMS)', 'measured.lockin_current_a_rms'),
                 ('SINE OUT readback (V RMS)', 'actual.lockin_excitation_v_rms')),
        value=None, description='X axis:',
    )
display(excitation_x_widget)
PLOT_X = excitation_x_widget.value or X
Y = "measured.smu_bias_current_a"
GROUP_BY = ("requested.lockin_excitation_v_rms",)
PLOT_GROUP_BY = tuple(
    field for field in GROUP_BY
    if not (PLOT_X in ('measured.lockin_current_a_rms', 'actual.lockin_excitation_v_rms')
            and field == 'requested.lockin_excitation_v_rms')
)
FILTERS = {}  # Example: {"requested.temperature_k": 2.0}
SORT_X = False  # Explicit view-only sorting; original sequence remains recorded.

selection = ()
figure = None
if not loaded_rows:
    print("Load records first.")
else:
    try:
        chosen = [row for row in loaded_rows if sample_id(row) not in excluded.value]
        selection = select_series(chosen, x=PLOT_X, y=Y, group_by=PLOT_GROUP_BY,
                                  filters=FILTERS, sort_x=SORT_X, audit=loaded_audit)
        if not selection:
            print("No matching finite observations. Check available columns/channels.")
        else:
            figure, axis = plot_series(selection, x=PLOT_X, y=Y)
            display(figure)
            import matplotlib.pyplot as plt
            plt.close(figure)
    except ValueError as error:
        print(error)


## 可选导出

只导出当前选择，不覆盖旧目录。所选数据、精确 sample IDs、自动分组键、筛选配置与校验值会随图保存。更改筛选后先重跑上一 cell；图不是论文投稿规范认证。


In [ ]:
EXPORT = False
EXPORT_DIRECTORY = PROJECT_ROOT / "analysis_output" / "combination_selection_01"
if EXPORT and selection and figure is not None:
    target = export_selection(EXPORT_DIRECTORY, selection, x=PLOT_X, y=Y,
                              group_by=PLOT_GROUP_BY, filters=FILTERS,
                              sort_x=SORT_X, audit=loaded_audit,
                              excluded_sample_ids=excluded.value)
    figure.savefig(target / "observations.png", dpi=600, facecolor="white")
    figure.savefig(target / "observations.pdf", facecolor="white")
    print(target)
